# Molecular Design VAE — Large GPU Mode

**~100,000 ZINC molecules · 400 epochs · GPU · ~20-40 min**

Best version of the model. Same data and architecture as Large CPU mode, but trains in 20-40 min on a free Colab T4 GPU.

### Before running:
1. **Switch runtime to GPU**: Runtime → Change runtime type → T4 GPU
2. **Get an ngrok token**: https://dashboard.ngrok.com/get-started/your-authtoken

## Cell 1 — Setup + verify GPU

In [ ]:
!git clone https://github.com/Kaur-Simarpreet/molecular-design-vae.git
%cd molecular-design-vae
!pip install -q torch selfies flask flask-cors scipy pyngrok requests
!pip install -q rdkit

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete')

## Cell 1b — Set up real DiffDock docking (~10 min, ~3 GB)

Optional but recommended for Large GPU mode. Downloads DiffDock + ESM-2 weights for state-of-the-art blind docking. Without this, scoring uses mock estimates.

In [ ]:
!bash setup_docking.sh --diffdock
!python docking.py   # verify


## Cell 2 — Train (20-40 min on T4 GPU)

First run downloads ZINC-250K (~8 min, then cached).

In [ ]:
!python train_vae_extended.py --mode large-gpu

## Cell 3 — Serve via ngrok

In [ ]:
from pyngrok import ngrok, conf
import threading, subprocess, time

NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"
conf.get_default().auth_token = NGROK_TOKEN

def run_server():
    subprocess.run(['python', 'serve.py'])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(10)

public_url = ngrok.connect(5000)
print(f'\n=== UI available at: {public_url} ===\n')
print('Trained on Colab T4 GPU. The model is bigger (4.9M params) but the UI is identical.')

## Cell 4 — Download trained model (4.9M params, ~20MB)

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('saved_model', 'zip', 'saved_model')
files.download('saved_model.zip')